# ***Industrial Vision SLM Benchmark***

## ***Notebook 00 — Préparation de l'environnement***

**Objectif :** vérifier l'environnement Python, la disponibilité
du GPU NVIDIA T4 et les dépendances nécessaires au projet.


## **1. Vérification de l'environnement Python et GPU**

In [ ]:
import sys
import torch

print("Python :", sys.version.split()[0])
print("PyTorch :", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

    props = torch.cuda.get_device_properties(0)
    print(f"VRAM totale : {props.total_memory / 1024**3} Gio")
else:
    print("Aucun GPU CUDA détecté.")

Python : 3.13.16
PyTorch : 2.14.1+cu130
CUDA disponible: True
GPU : Tesla T4
VRAM totale : 14.56317138671875 Gio


## **2. Test de calcul matriciel sur GPU**

In [ ]:
import torch

# 1. Sélection du GPU CUDA
device = torch.device("cuda")

# 2. Création de deux matrices de 1024x1024 directement dans le GPU
A = torch.randn(1024, 1024, device=device)
B = torch.randn(1024, 1024, device=device)

# 3. Multiplication matricielle sur le GPU
C = A @ B

# 4. Attente de la fin des calculs
torch.cuda.synchronize()

# 5. Vérif du périphérique et du résultat
print("GPU utilisé :", C.device)
print("DImensions du résultat :", C.shape)
print("Résultat valide :",torch.isfinite(C).all().item())

# 6. Mesure de la mémoire GPU occupée par les tenseurs PyTorch
memory = torch.cuda.memory_allocated() / 1024**2
print(f"Memoire GPU allouée : {memory:.2f} Mio")

# 7. Libération des mtrices devenues inutiles
del A, B, C
torch.cuda.empty_cache()

GPU utilisé : cuda:0
DImensions du résultat : torch.Size([1024, 1024])
Résultat valide : True
Memoire GPU allouée : 20.12 Mio


## **3. Vérification des bibliothèques nécessaires**

### *3.1. Inventaire des dépendances Python*

In [ ]:
from importlib.metadata import version, PackageNotFoundError

# Bibliothèques nécessaires au projet
packages = [
    "torch", "transformers", "accelerate", "peft",
    "bitsandbytes", "datasets", "sentence-transformers",
    "faiss-cpu", "pypdf", "trl", "streamlit",
]

print("=== Vérification des dépendances ===\n")

for package in packages:
    try:
        # Récuperer la version installée
        installed_version = version(package)
        print(f"[OK] {package:<24} {installed_version}")
    except PackageNotFoundError:
        print(f"[--] {package:<24} Non installé")

print("\nVérification terminée.")

=== Vérification des dépendances ===

[OK] torch                    2.14.1+cu130
[OK] transformers             5.22.1
[OK] accelerate               1.15.0
[OK] peft                     0.21.2
[--] bitsandbytes             Non installé
[OK] datasets                 4.8.5
[OK] sentence-transformers    5.7.0
[--] faiss-cpu                Non installé
[--] pypdf                    Non installé
[--] trl                      Non installé
[--] streamlit                Non installé

Vérification terminée.


## **4. Installation des dépendances compatibles**

### *4.1. Installation des bibliothèques manquantes*

In [ ]:
%pip install bitsandbytes faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 125.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [bitsandbytes]


### *4.2. Vérification des dépendances installées*

In [ ]:
from importlib.metadata import version
import torch
import faiss
import bitsandbytes
import pypdf

# Affichage des versions des bibliothèques
packages = [
    "torch", "bitsandbytes", "faiss-cpu", "pypdf",
]

print("=== Versions installées ===\n")

for package in packages:
    print(f"{package:<16} : {version(package)}")

# Vérifier que PyTorch détacte toujours notre GPU
print("\n=== Vérification GPU ===\n")

print("CUDA disponible :", torch.cuda.is_available())
print("Version CUDA PyTorch :", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

print("\nVérification terminée.")

=== Versions installées ===

torch            : 2.14.1+cu130
bitsandbytes     : 0.50.2
faiss-cpu        : 1.15.1
pypdf            : 7.0.0

=== Vérification GPU ===

CUDA disponible : True
Version CUDA PyTorch : 13.0
GPU : Tesla T4

Vérification terminée.


## **5. Connexion à Google Drive**

### *5.1. Montage et vérification de Google Drive*

In [ ]:
from google.colab import drive
from pathlib import Path

# 1. Connexion Google Drive à notre environnement Colab
drive.mount("/content/drive")

# 2. Définnition du chemin racine de notre espace Drive
drive_root = Path("/content/drive/MyDrive")

# 3. Vérif que le répertoire est accessible
print("\n=== Vérification Google Drive ===\n")

print("Chemin :", drive_root)
print("Drive accessible :", drive_root.is_dir())

if drive_root.is_dir():
    print("[OK] Google Drive est correctement monté.")
else:
    print("[ERREUR] Google Drive est inacessible.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

=== Vérification Google Drive ===

Chemin : /content/drive/MyDrive
Drive accessible : True
[OK] Google Drive est correctement monté.


## **6. Préparation de la structure du projet**

### *6.1. Création des répertoires de stockage*

In [ ]:
from pathlib import Path

# 1. Définition du répertoire principal du projet sur Drive
PROJECT_ROOT = Path("/content/drive/MyDrive/industrial-vision-slm-benchmark")

# 2. Définir les sous-répertoires nécessaires
directories = [
    "data/raw", "data/processed", "data/instructions", "data/preferences", "data/evaluations",
    "outputs/vector_index", "outputs/checkpoints", "outputs/adapters", "outputs/metrics", "outputs/figures",
]

# 3. Création des répertoires
for directory in directories:
    path = PROJECT_ROOT / directory
    path.mkdir(parents=True, exist_ok=True)

    print(f"[OK] {directory}")

print("\nRépertoire principal :", PROJECT_ROOT)
print("Nombre de sous-répertoires :", len(directories))

[OK] data/raw
[OK] data/processed
[OK] data/instructions
[OK] data/preferences
[OK] data/evaluations
[OK] outputs/vector_index
[OK] outputs/checkpoints
[OK] outputs/adapters
[OK] outputs/metrics
[OK] outputs/figures

Répertoire principal : /content/drive/MyDrive/industrial-vision-slm-benchmark
Nombre de sous-répertoires : 10


### *6.2. Vérification de l'arborescence*

In [ ]:
missing = []

for directory in directories:
    path = PROJECT_ROOT / directory
    if not path.is_dir():
        missing.append(directory)

if not missing:
    print("[OK] Tous les répertoires existent.")
    print(f"[OK] {len(directories)} sous-répertoires vérifiés.")
else:
    print("[ERREUR] Répertoires manquants :")
    for directory in missing:
        print(" -", directory)

[OK] Tous les répertoires existent.
[OK] 10 sous-répertoires vérifiés.


### *6.3. Récupération du dépôt GitHub dans Colab*

In [ ]:
from pathlib import Path
import subprocess

# 1. Adresse du dépôt GitHub
REPO_URL = "https://github.com/I-am-Axel-Helios/industrial-vision-slm-benchmark"

# 2. Emplacement du code dans Colab
REPO_PATH = Path("/content/industrial-vision-slm-benchmark")

# 3. Clonage du dépôt uniquement s'il n'existe pas
if not REPO_PATH.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_PATH)], check=True)
    print(f"[OK] Repo Github cloné")
elif (REPO_PATH / ".git").is_dir():
    print("[INFO] Repo déjà présent.")

    # Récupérer les nouveaux commits sans créer de fusion
    subprocess.run(["git", "-C", str(REPO_PATH), "pull", "--ff-only"], check=True)
else:
    raise RuntimeError("Dossier existatnt, mais pas un repo Git.")

# 4. Vérif de l'existence du répertoire
print("\nRépertoire :", REPO_PATH)
print("Dépôt accessible :", REPO_PATH.is_dir())

# 5. Affichage des deux derniers commits
print("\n=== Historique Git ===\n")
result = subprocess.run(
    ["git", "-C", str(REPO_PATH), "log", "-2", "--oneline"],
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout)

[INFO] Repo déjà présent.

Répertoire : /content/industrial-vision-slm-benchmark
Dépôt accessible : True

=== Historique Git ===

98f7442 docs: add Colab environment setup notebook
778169e chore: initialize project structure and configuration



## **7. Découverte du tokenizer Qwen3-1.7B**

### *7.1. Chargement du tokenizer Hugging Face*

In [ ]:
from transformers import AutoTokenizer

# 1. Identifiant du modèle sur Hugging Face
MODEL_ID = "Qwen/Qwen3-1.7B"

# 2. Chargement du tokenizer préentraîné
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 3. Affichage des caractéristiques
print("=== Tokenizer Qwen3===\n")

print("Modèle :", MODEL_ID)
print("Classe :", type(tokenizer).__name__)
print("Taille du vocabulaire :", tokenizer.vocab_size)
print("Token EOS :", tokenizer.eos_token_id)
print("ID du token EOS :", tokenizer.eos_token_id)
print("Token PAD :", tokenizer.pad_token_id)

print("\n[OK] Tokenizer chargé avec succès")

=== Tokenizer Qwen3===

Modèle : Qwen/Qwen3-1.7B
Classe : Qwen2Tokenizer
Taille du vocabulaire : 151643
Token EOS : 151645
ID du token EOS : 151645
Token PAD : 151643

[OK] Tokenizer chargé avec succès


### *7.2. Tokenisation d'une question technique*

In [ ]:
# 1. Question liée à notre domaine de spécialisation
question = "Comment supprimer le bruit impulsionnel dans une image avec OpenCV ?"

# 2. Conversion en ids de tokens
token_ids = tokenizer.encode(question, add_special_tokens=False)

# 3. Récup de la représentation textuelle des tokens
tokens = tokenizer.convert_ids_to_tokens(token_ids)

# 4. Affichage des résultats
print("=== Question ===\n")
print(question)

print("\n=== Tokens & identifiants ===\n")

for index, (token, token_id) in enumerate(zip(tokens, token_ids)):
    print(f"{index:02d} | {token!r:<18} | ID : {token_id}")

print("\nNombre total de tokens :", len(token_ids))

# 5. Décodage des identifiants vers du texte
decoded_text = tokenizer.decode(token_ids, skip_special_tokens=False)

print("\n=== Décodage ===\n")
print(decoded_text)

# 6. Verif que le texte est correctement reconstruit
print("\nReconstruction exacte :", decoded_text == question)

=== Question ===

Comment supprimer le bruit impulsionnel dans une image avec OpenCV ?

=== Tokens & identifiants ===

00 | 'Comment'          | ID : 10677
01 | 'Ġsup'             | ID : 1026
02 | 'primer'           | ID : 47570
03 | 'Ġle'              | ID : 512
04 | 'Ġbr'              | ID : 1411
05 | 'uit'              | ID : 3083
06 | 'Ġimp'             | ID : 3163
07 | 'ulsion'           | ID : 32903
08 | 'nel'              | ID : 8138
09 | 'Ġdans'            | ID : 6866
10 | 'Ġune'             | ID : 6185
11 | 'Ġimage'           | ID : 2168
12 | 'Ġavec'            | ID : 9753
13 | 'ĠOpen'            | ID : 5264
14 | 'CV'               | ID : 19589
15 | 'Ġ?'               | ID : 937

Nombre total de tokens : 16

=== Décodage ===

Comment supprimer le bruit impulsionnel dans une image avec OpenCV ?

Reconstruction exacte : True


## **8. Chargement du SLM Qwen3-1.7B**

### *8.1. Chargement du modèle en FP16 sur GPU*

In [ ]:
import torch
from transformers import AutoModelForCausalLM

# 1. Identifiant du modèle sur Hugging Face
MODEL_ID = "Qwen/Qwen3-1.7B"

# 2. Vérif de la disponiilité du GPU
assert torch.cuda.is_available(), "Aucun GPU CUDA disponible"

print("=== Chargement du SLM ===\n")

print("Modèle :", MODEL_ID)
print("GPU :", torch.cuda.get_device_name(0))

# 3. Chargement du modèle en précision FP16
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")

# 4. Activation du mode évaluation
model.eval()

# 5. Affichage des caractéristiques du modèle
print("\n=== Caractéristiques ===\n")

print("Architecture :", type(model).__name__)
print("Type :", model.config.model_type)
print("Device :", model.device)
print("Précision :", model.dtype)

# 6. Nombre de paramètres total
num_parameters = sum(p.numel() for p in model.parameters())
print(f"Paramètres : {num_parameters / 1e9:.2f} milliards")


# 7. Mémoire GPU
allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3

print("\n=== Mémoire GPU ===\n")
print(f"VRAM allouée : {allocated:.2f} Gio")
print(f"VRAM réservée : {reserved:.2f} Gio")


print("\n[OK] SLM chargé avec succès")

=== Chargement du SLM ===

Modèle : Qwen/Qwen3-1.7B
GPU : Tesla T4


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]


=== Caractéristiques ===

Architecture : Qwen3ForCausalLM
Type : qwen3
Device : cuda:0
Précision : torch.float16
Paramètres : 1.72 milliards

=== Mémoire GPU ===

VRAM allouée : 3.21 Gio
VRAM réservée : 7.34 Gio

[OK] SLM chargé avec succès


### *8.2. Première inférence — Question sur OpenCV*

In [ ]:
import torch

# 1. Définition de notre question technique
question = (
    "Quelle est la différence entre un filtre gaussien et un filtre médian ?"
    "Lequel est le plus adapté pour supprimer du bruit impulsionnel dans une image ?"
)

# 2. Construction des messages de conversation
messages = [
    {
        "role": "system",
        "content": (
            "Tu es un assistant spécialisé en traitement d'images. "
            "Réponds en français, de manière précise et concise. "
            "Compare les méthodes demandées, puis donne une conclusion. "
            "Évite les détails inutiles et limite-toi à 150 mots environ. "
            "Si tu ne connais pas une réponse, indique-le."
        )
    },
    {
        "role": "user",
        "content": question
    }
]

# 3. Appliquer le format conversationnel de Qwen3
formatted_prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)

# 4. Conversion du prompt en tenseurs PyTorch
inputs = tokenizer([formatted_prompt], return_tensors="pt").to(model.device)

# 5. Génération de la réponse (sans calcul de gradients)
with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=512, do_sample=False , pad_token_id=tokenizer.eos_token_id)

# 6. Conservation uniquement des tokens générés
input_length = inputs["input_ids"].shape[1]
generated_ids = outputs[0][input_length:]

# 7. Décodage des tokens en texte
response = tokenizer.decode(generated_ids, skip_special_tokens=True)

# 8. Affichage de la réponse
print("\n=== Réponse de Qwen3-1.7B ===\n")
print(response)

print("\nTokens générés :", len(generated_ids))
print("\n[OK] Première inférence terminée")

# Récupérer l'identifiant du dernier token généré.
last_token_id = generated_ids[-1].item()

# Vérifier si le modèle a émis son token de fin.
finished_normally = (
    last_token_id == tokenizer.eos_token_id
)

print("\n=== Vérification de la génération ===\n")

print("Tokens générés :", len(generated_ids))
print("Fin de séquence détectée :", finished_normally)

if finished_normally:
    print("[OK] Le modèle a terminé sa génération.")
elif len(generated_ids) >= 512:
    print("[ATTENTION] Limite de tokens atteinte.")
else:
    print("[INFO] Génération interrompue sans token EOS.")



=== Réponse de Qwen3-1.7B ===

Un filtre gaussien utilise une distribution gaussienne pour éliminer le bruit, ce qui est efficace pour le bruit gaussien ou blanc, tandis qu'un filtre médian est plus adapté pour le bruit impulsionnel, car il élimine les valeurs extrêmes sans distorsion. Le filtre médian est donc le plus adapté pour supprimer du bruit impulsionnel.

Tokens générés : 87

[OK] Première inférence terminée

=== Vérification de la génération ===

Tokens générés : 87
Fin de séquence détectée : True
[OK] Le modèle a terminé sa génération.


### *8.3. Mesure de la latence et de la mémoire GPU*

In [ ]:
import time
import torch

# 1. Verification de la disponibilité du GPU
assert torch.cuda.is_available(), "GPU CUDA indisponible"

# 2. Attente de la fin des éventuelles opérations CUDA
torch.cuda.synchronize()

# 3. Réinitialisation des stats de mémoire maximale
torch.cuda.reset_peak_memory_stats()

# 4. Démarrage chronomètre
start_time = time.perf_counter()

# 5. Génération de la réponse
with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.eos_token_id)

# 6. Attente de la fin des calculs GPU
torch.cuda.synchronize()

# 7. Arrêt du chronomètre
end_time = time.perf_counter()

# 8. Durée de génération
latency = end_time - start_time

# 9. Décompte des tokens générés
input_length = inputs["input_ids"].shape[1]
generated_ids = outputs[0, input_length:]

num_tokens = len(generated_ids)

# 10. Vitesse moyenne
tokens_per_second = num_tokens / latency

# 11. Mémoire GPU
peak_memory = torch.cuda.max_memory_allocated() / 1024**3
current_memory = torch.cuda.memory_allocated() / 1024**3

# 12. Résultats
print("=== Perfs de Qwen3-1.7B ===\n")

print(f"Temps d'inférence : {latency:.2f} s")
print(f"Tokens générés : {num_tokens}")
print(f"Vitesse moyenne : {tokens_per_second:.2f} tokens/s")

print("\n=== Mémoire GPU ===\n")
print(f"VRAM actuellement allouée : {current_memory:.2f} Gio")
print(f"pic de VRAM allouée : {peak_memory:.2f} Gio")

print("\n[OK] Mesure de latence et de mémoire terminée")

=== Perfs de Qwen3-1.7B ===

Temps d'inférence : 5.29 s
Tokens générés : 87
Vitesse moyenne : 16.44 tokens/s

=== Mémoire GPU ===

VRAM actuellement allouée : 3.21 Gio
pic de VRAM allouée : 3.24 Gio

[OK] Mesure de latence et de mémoire terminée


### *8.4. Répétition des mesures d'inférence*

**Objectif :** vérifier la stabilité du temps d'inférence,
du débit de génération et de la consommation mémoire.

**Protocole :**
- Même modèle Qwen3-1.7B en FP16.
- Même question technique.
- Génération déterministe.
- Trois exécutions successives.
- Calcul de la médiane des performances.

In [28]:
import time
import statistics
import torch

# 1. Nombre de répétitions
N_RUNS = 3

# 2. Listes pour conserver les résultats
latencies = []
throughputs = []
peak_memories = []

print("=== Benchmark répété de Qwen3-1.7B ===\n")

# 3. Répéter exactement la même inférence
for run in range(N_RUNS):
    # Synchronisation du GPU avant chronométrage
    torch.cuda.synchronize()

    # Réinitialisation des stats de mémoire
    torch.cuda.reset_peak_memory_stats()

    # Démarrage du chronomètre
    start_time = time.perf_counter()

    # Génération de la réponse
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.eos_token_id)

    # Attente de la fin des calculs GPU
    torch.cuda.synchronize()

    # Calcul de la durée de génération
    latency = time.perf_counter() - start_time

    # Décompte des nouveaux tokens
    input_length = inputs["input_ids"].shape[1]
    num_tokens = outputs.shape[1] - input_length

    # Débit
    throughput = num_tokens / latency

    # Pic de VRAM
    peak_memory = torch.cuda.max_memory_allocated() / 1024**3

    # Enregistrement des résultats
    latencies.append(latency)
    throughputs.append(throughput)
    peak_memories.append(peak_memory)

    print(
        f"Test {run + 1} : "
        f"Temps d'inférence : {latency:.2f} s | "
        f"Tokens générés : {num_tokens} tokens | "
        f"Vitesse moyenne : {throughput:.2f} tokens/s | "
        f"VRAM : {peak_memory:.2f} Gio"
    )

# 4. Calcul des stats récap
print("\n=== Résultats récapitulatifs ===\n")
print(f"Latence médiane : {statistics.median(latencies):.2f} s")
print(f"Débit médian : {statistics.median(throughputs):.2f} tokens/s")
print(f"Pic VRAM maximal : {max(peak_memories):.2f} Gio")

print("\n[OK] Benchmark répété terminé")

=== Benchmark répété de Qwen3-1.7B ===

Test 1 : Temps d'inférence : 3.32 s | Tokens générés : 87 tokens | Vitesse moyenne : 26.17 tokens/s | VRAM : 3.24 Gio
Test 2 : Temps d'inférence : 3.77 s | Tokens générés : 87 tokens | Vitesse moyenne : 23.06 tokens/s | VRAM : 3.24 Gio
Test 3 : Temps d'inférence : 3.86 s | Tokens générés : 87 tokens | Vitesse moyenne : 22.51 tokens/s | VRAM : 3.24 Gio

=== Résultats récapitulatifs ===

Latence médiane : 3.77 s
Débit médian : 23.06 tokens/s
Pic VRAM maximal : 3.24 Gio

[OK] Benchmark répété terminé


### *8.5. Sauvegarde des résultats d'inférence*

**Objectif :** conserver les mesures de latence, de débit
et de mémoire GPU dans un fichier JSON sur Google Drive.

Les résultats serviront de référence pour comparer les
futures versions du SLM.

In [30]:
import json
import statistics
from pathlib import Path
from datetime import datetime, timezone

# 1. Répertoire persistant des métriques
METRICS_DIR = Path("/content/drive/MyDrive/industrial-vision-slm-benchmark/outputs/metrics")

# 2. Verif de l'accessibilité de Google Drive
if not METRICS_DIR.is_dir():
    raise FileNotFoundError("Répertoire des métriques non accessible. Vérifie le montage de Google Drive (section 5.1).")

# 3. Regroupement des résultats de chaque exécution
runs = []

for i, (latency, throughput, memory) in enumerate(zip(latencies, throughputs, peak_memories)):
    runs.append({
        "run": i + 1,
        "latency_s": round(latency, 3),
        "tokens_per_s": round(throughput, 3),
        "peak_vram_gib": round(memory, 3)
    })

# 4. Construction du rapport
timestamp = datetime.now(timezone.utc)

report = {
    "timestamp_utc": timestamp.isoformat(),
    "model": "Qwen/Qwen3-1.7B",
    "gpu": torch.cuda.get_device_name(0),
    "precision": "float16",
    "question": question,
    "generation_config": {
        "max_new_tokens": 512,
        "do_sample": False,
        "pad_token_id": tokenizer.eos_token_id
    },
    "input_tokens": int(inputs["input_ids"].shape[1]),
    "output_tokens": int(num_tokens),
    "runs": runs,
    "summary": {
        "median_latency_s": round(statistics.median(latencies), 3),
        "median_tokens_per_second": round(statistics.median(throughputs), 3),
        "max_peak_vram_gib": round(max(peak_memories), 3)
    }
}

# 5. Création d'un nom unique pour éviter les écrasements
filename = "qwen3-1.7b_fp16_" + timestamp.strftime("%Y%m%d_%H%M%S") + "_utc.json"
output_path = METRICS_DIR / filename

# 6. Sauvegarde du rapport
output_path.write_text(
    json.dumps(report, indent=4, ensure_ascii=False),
    encoding="utf-8"
)

# 7. Vérif du fichier sauvegardé
print("=== Sauvegarde des métriques ===\n")

print("Fichier :", output_path)
print("Fichier existant :", output_path.is_file())
print("Nombre de tests :", len(runs))

print("\n[OK] Résultats sauvegardés sur Google Drive")

=== Sauvegarde des métriques ===

Fichier : /content/drive/MyDrive/industrial-vision-slm-benchmark/outputs/metrics/qwen3-1.7b_fp16_20261109_201557_utc.json
Fichier existant : True
Nombre de tests : 3

[OK] Résultats sauvegardés sur Google Drive
